In [1]:
from google.colab import files, drive
files.upload()

Saving test_data (2).csv to test_data (2) (1).csv
Saving train_data (2).csv to train_data (2) (1).csv


{'test_data (2) (1).csv': b'id,name,make,model,year,engine,cylinders,fuel,mileage,transmission,trim,body,doors,exterior_color,interior_color,drivetrain\r\n803,2023 Dodge Durango Pursuit,Dodge,Durango,2023,16V MPFI OHV,8,Gasoline,32,8-Speed Automatic,Pursuit,SUV,4,DB Black Crystal Clearcoat,Black,All-wheel Drive\r\n804,2024 Ford Edge SEL,Ford,Edge,2024,"t 2L I-4 gasoline direct injection, DOHC, variable valve",4,Gasoline,10,Automatic,SEL,SUV,4,Carbonized Gray Metallic,Dune,All-wheel Drive\r\n805,2024 Chevrolet Equinox FWD RS,Chevrolet,Equinox,2024,"4 gasoline direct injection, DOHC, variable valve control",4,Gasoline,2,6-Speed Automatic Electronic with Overdrive,FWD RS,SUV,4,Summit White,Jet Black With Red Accents,Front-wheel Drive\r\n806,2024 Hyundai Tucson Hybrid N Line,Hyundai,Tucson Hybrid,2024,"ream 1.6L I-4 gasoline direct injection, DOHC, CVVD varia",4,Hybrid,10,Automatic,N Line,SUV,4,Titan Gray,Black w/Red Accents,All-wheel Drive\r\n807,2024 Jeep Wagoneer Base,Jeep,Wagoneer,2024

In [2]:
!ls

 sample_data		 'test_data (2).csv'	   'train_data (2).csv'
'test_data (2) (1).csv'  'train_data (2) (1).csv'


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
train_df = pd.read_csv('train_data (2).csv')
test_df = pd.read_csv('test_data (2).csv')

In [6]:
train_df.head()

,id,name,make,model,year,price,engine,cylinders,fuel,mileage,transmission,trim,body,doors,exterior_color,interior_color,drivetrain
0,1,2024 INFINITI QX50 SPORT,INFINITI,QX50,2024,49404.0,ER,4.0,Gasoline,7.0,(CVT) CONT VAR.,SPORT,NaN,4.0,Radiant White,Monaco Red,Front-wheel Drive
1,2,2024 Mazda CX-90 PHEV Base,Mazda,CX-90 PHEV,2024,55399.0,"TIV-G 2.5L I-4 gasoline direct injection, DOHC...",4.0,Gasoline,2.0,8-Speed Automatic,Base,SUV,4.0,Soul Red Crystal Metallic,Griege,All-wheel Drive
2,3,2024 Honda CR-V Hybrid Sport Touring,Honda,CR-V Hybrid,2024,42005.0,16V GDI DOHC Hybrid,4.0,Hybrid,1.0,Automatic CVT,Sport Touring,SUV,4.0,Radiant Red Metallic,Black,All-wheel Drive
3,4,2024 Jeep Grand Cherokee L Limited,Jeep,Grand Cherokee L,2024,52567.0,OHV,6.0,Gasoline,246.0,8-Speed Automatic,Limited,SUV,4.0,Velvet Red Pearlcoat,Global Black,Four-wheel Drive
4,5,2024 Acura MDX Technology,Acura,MDX,2024,56800.0,24V GDI SOHC,6.0,Gasoline,5.0,10-Speed Automatic,Technology,SUV,4.0,Gray,NaN,Front-wheel Drive


In [5]:
# Subtask 1
train_df.groupby(by='cylinders')['price'].mean()

,price
cylinders,
0.0,66045.000000
3.0,32673.421053
4.0,41099.911051
6.0,60456.972973
8.0,58899.587302


In [6]:
subtask_1 = pd.DataFrame({
    'subtaskID': [1],
    'datapointID': [1],
    'answer': train_df.groupby(by='cylinders')['price'].mean()[0],
})
subtask_1.head()

,subtaskID,datapointID,answer
0,1,1,66045.0


In [29]:
train_df.head()

,id,name,make,model,year,price,engine,cylinders,fuel,mileage,transmission,trim,body,doors,exterior_color,interior_color,drivetrain
0,1,2024 INFINITI QX50 SPORT,INFINITI,QX50,2024,49404.0,ER,4.0,Gasoline,7.0,(CVT) CONT VAR.,SPORT,NaN,4.0,Radiant White,Monaco Red,Front-wheel Drive
1,2,2024 Mazda CX-90 PHEV Base,Mazda,CX-90 PHEV,2024,55399.0,"TIV-G 2.5L I-4 gasoline direct injection, DOHC...",4.0,Gasoline,2.0,8-Speed Automatic,Base,SUV,4.0,Soul Red Crystal Metallic,Griege,All-wheel Drive
2,3,2024 Honda CR-V Hybrid Sport Touring,Honda,CR-V Hybrid,2024,42005.0,16V GDI DOHC Hybrid,4.0,Hybrid,1.0,Automatic CVT,Sport Touring,SUV,4.0,Radiant Red Metallic,Black,All-wheel Drive
3,4,2024 Jeep Grand Cherokee L Limited,Jeep,Grand Cherokee L,2024,52567.0,OHV,6.0,Gasoline,246.0,8-Speed Automatic,Limited,SUV,4.0,Velvet Red Pearlcoat,Global Black,Four-wheel Drive
4,5,2024 Acura MDX Technology,Acura,MDX,2024,56800.0,24V GDI SOHC,6.0,Gasoline,5.0,10-Speed Automatic,Technology,SUV,4.0,Gray,NaN,Front-wheel Drive


In [34]:
# Subtask 2 - regression model to predict the `price` for all cars in `test_data.csv`
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

train_df = train_df.dropna(subset=['price'])

X_train = train_df.drop(columns=['id', 'price'])
y_train = train_df['price']

X_test = test_df.drop(columns=['id'])

cat_cols = X_train.select_dtypes(exclude='number').columns
num_cols = X_train.select_dtypes(include='number').columns

trans = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'
)

pipe = Pipeline([
    ('trans', trans),
    ('regressor', XGBRegressor(n_estimators=250, n_jobs=-1, random_state=42, missing=np.nan, device='cuda'),)
], verbose=True)

pipe.fit(X_train, y_train)

[Pipeline] ............. (step 1 of 2) Processing trans, total=   0.0s


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [18:14:37] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [18:14:37] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[Pipeline] ......... (step 2 of 2) Processing regressor, total=   0.4s


/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('trans',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['name', 'make', 'model', 'engine', 'fuel', 'transmission', 'trim',
       'body', 'drivetrain'],
      dtype='object'))])),
                ('regressor',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=No...
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=250, n_jobs=-1,
                              num_parallel_tree=None, ...))],
         verbose=True)

In [40]:
# Subtask 2/Alt (with TargetEncoder) - regression model to predict the `price` for all cars in `test_data.csv`
from xgboost import XGBRegressor
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

train_df = train_df.dropna(subset=['price'])

X_train = train_df.drop(columns=['id', 'price'])
y_train = train_df['price']

X_test = test_df.drop(columns=['id'])

# We define the columns with high cardinality (many unique text values)
high_card_cols = ['name', 'make', 'model', 'fuel', 'engine', 'transmission', 'trim', 'body', 'exterior_color', 'interior_color', 'drivetrain']
num_cols = ['year', 'cylinders', 'mileage', 'doors']

preprocessor = ColumnTransformer(
    transformers=[
        ('target', TargetEncoder(smooth='auto', random_state=42), high_card_cols)
    ],
    remainder='passthrough'
)

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_estimators=500, n_jobs=-1, random_state=42, missing=np.nan, device='cuda'),)
], verbose=True)

pipe.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


[Pipeline] ...... (step 1 of 2) Processing preprocessor, total=   1.8s


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [18:21:53] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [18:21:53] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[Pipeline] ......... (step 2 of 2) Processing regressor, total=  20.8s


/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('target',
                                                  TargetEncoder(random_state=42),
                                                  ['name', 'make', 'model',
                                                   'fuel', 'engine',
                                                   'transmission', 'trim',
                                                   'body', 'exterior_color',
                                                   'interior_color',
                                                   'drivetrain'])])),
                ('regressor',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsampl...
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=-1,
                              num_parallel_tree=None, ...))],
         verbose=True)

In [41]:
y_preds = pipe.predict(X_test)

In [42]:
abs(y_preds.std() - train_df['price'].std()) # Seems pretty similar

np.float32(4143.46)

In [43]:
subtask_2 = pd.DataFrame({
    'subtaskID': 2,
    'datapointID': test_df['id'],
    'answer': y_preds,
})
subtask_2.head()

,subtaskID,datapointID,answer
0,2,803,46691.414062
1,2,804,39232.628906
2,2,805,31104.685547
3,2,806,37593.027344
4,2,807,67810.296875


In [44]:
submission = pd.concat([subtask_1, subtask_2])
submission.to_csv('submission.csv', index=False)

files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>